# Reservas Técnicas en Seguros

## Métodos: Chain Ladder, Bornhuetter-Ferguson y Bootstrap

Las reservas técnicas son estimaciones de las obligaciones futuras de la aseguradora. Este notebook demuestra los principales métodos actuariales para calcular reservas.

### Contenido
1. Introducción a las reservas técnicas
2. Carga de triángulos de desarrollo
3. Método Chain Ladder
4. Método Bornhuetter-Ferguson
5. Método Bootstrap (intervalos de confianza)
6. Comparación de métodos
7. Visualizaciones y análisis de sensibilidad

In [ ]:
# Imports necesarios
from decimal import Decimal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports de reservas
from mexican_insurance.reservas import (
    ChainLadder,
    BornhuetterFerguson,
    Bootstrap,
    TrianguloDesarrollo
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports completados exitosamente")

## 1. Introducción a las Reservas Técnicas

Las reservas técnicas representan el valor presente de las obligaciones futuras de la aseguradora:

- **Reserva de Siniestros Pendientes**: Para siniestros reportados pero no liquidados (IBNR)
- **Reserva IBNR**: Para siniestros incurridos pero no reportados
- **Reserva de Riesgos en Curso**: Para la porción no devengada de las primas

### Métodos principales:
1. **Chain Ladder**: Basado en patrones históricos de desarrollo
2. **Bornhuetter-Ferguson**: Combina desarrollo histórico con prima esperada
3. **Bootstrap**: Método estocástico para intervalos de confianza

## 2. Carga de Triángulo de Desarrollo

Un triángulo de desarrollo muestra la evolución de los siniestros a lo largo del tiempo.

In [ ]:
# Cargar triángulo de desarrollo desde CSV
triangulo_df = pd.read_csv('../examples/data/triangulo_ejemplo.csv', index_col=0)

print("📊 TRIÁNGULO DE DESARROLLO")
print("="*70)
print(f"Años de origen: {len(triangulo_df)}")
print(f"Períodos de desarrollo: {len(triangulo_df.columns)}")
print(f"\nTriángulo de siniestros acumulados:\n")
print(triangulo_df.to_string())

# Crear objeto TrianguloDesarrollo
triangulo = TrianguloDesarrollo(triangulo_df)

print(f"\n✅ Triángulo cargado exitosamente")
print(f"Total siniestros pagados hasta la fecha: ${triangulo.total_pagado():,.0f}")

In [ ]:
# Visualizar el triángulo como heatmap
fig, ax = plt.subplots(figsize=(12, 8))

sns.heatmap(triangulo_df, annot=True, fmt='.0f', cmap='YlOrRd', 
            cbar_kws={'label': 'Siniestros Acumulados (MXN)'},
            linewidths=0.5, linecolor='gray', ax=ax)

ax.set_title('Triángulo de Desarrollo - Siniestros Acumulados', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Período de Desarrollo', fontsize=12)
ax.set_ylabel('Año de Origen', fontsize=12)

plt.tight_layout()
plt.show()

## 3. Método Chain Ladder

El método Chain Ladder asume que el patrón de desarrollo histórico se repetirá en el futuro. Calcula factores de desarrollo para proyectar los siniestros finales.

In [ ]:
# Aplicar Chain Ladder
cl = ChainLadder(triangulo)
resultado_cl = cl.calcular_reservas()

print("📊 MÉTODO CHAIN LADDER")
print("="*70)
print(f"\nFactores de Desarrollo (LDFs):")
for i, factor in enumerate(resultado_cl['factores_desarrollo']):
    print(f"  Período {i} -> {i+1}: {factor:.4f}")

print(f"\n\nSiniestros Últimos (Projected Ultimate):")
for año, ultimo in resultado_cl['siniestros_ultimos'].items():
    print(f"  Año {año}: ${ultimo:,.0f}")

print(f"\n\nReservas por Año de Origen:")
total_reserva_cl = 0
for año, reserva in resultado_cl['reservas'].items():
    print(f"  Año {año}: ${reserva:,.0f}")
    total_reserva_cl += reserva

print(f"\n{'='*70}")
print(f"RESERVA TOTAL (Chain Ladder): ${total_reserva_cl:,.0f}")
print(f"{'='*70}")

In [ ]:
# Visualizar factores de desarrollo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Factores de desarrollo
periodos = list(range(len(resultado_cl['factores_desarrollo'])))
factores = resultado_cl['factores_desarrollo']

ax1.plot(periodos, factores, marker='o', linewidth=2.5, markersize=10, 
         color='#3498db', label='LDF')
ax1.axhline(y=1.0, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Factor = 1.0')
ax1.set_xlabel('Período de Desarrollo', fontsize=12)
ax1.set_ylabel('Factor de Desarrollo', fontsize=12)
ax1.set_title('Factores de Desarrollo (LDF)\nChain Ladder', 
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Agregar valores
for i, (p, f) in enumerate(zip(periodos, factores)):
    ax1.text(p, f + 0.02, f'{f:.3f}', ha='center', va='bottom', 
             fontsize=10, fontweight='bold')

# Gráfica 2: Reservas por año
años_reserva = list(resultado_cl['reservas'].keys())
reservas_valores = list(resultado_cl['reservas'].values())

bars = ax2.bar(años_reserva, reservas_valores, color='#e74c3c', 
               alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Año de Origen', fontsize=12)
ax2.set_ylabel('Reserva (MXN)', fontsize=12)
ax2.set_title('Reservas por Año de Origen\nChain Ladder', 
              fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Agregar valores sobre las barras
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'${height/1e6:.2f}M',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Método Bornhuetter-Ferguson

El método B-F combina el desarrollo observado con una estimación a priori del siniestro último (basada en la prima esperada). Es más estable que Chain Ladder para años recientes con poco desarrollo.

In [ ]:
# Definir primas ganadas por año (necesarias para B-F)
primas_ganadas = {
    2018: 5000000,
    2019: 5500000,
    2020: 6000000,
    2021: 6500000,
    2022: 7000000,
    2023: 7500000,
}

# Loss ratio esperado (70%)
loss_ratio_esperado = 0.70

print("📊 CONFIGURACIÓN BORNHUETTER-FERGUSON")
print("="*70)
print(f"Loss Ratio Esperado: {loss_ratio_esperado*100}%")
print(f"\nPrimas Ganadas por Año:")
for año, prima in primas_ganadas.items():
    print(f"  {año}: ${prima:,.0f}")

In [ ]:
# Aplicar Bornhuetter-Ferguson
bf = BornhuetterFerguson(triangulo, primas_ganadas, loss_ratio_esperado)
resultado_bf = bf.calcular_reservas()

print("📊 MÉTODO BORNHUETTER-FERGUSON")
print("="*70)

print(f"\nSiniestros Últimos Esperados:")
for año, ultimo in resultado_bf['siniestros_ultimos_esperados'].items():
    print(f"  Año {año}: ${ultimo:,.0f}")

print(f"\n\nPorcentaje No Desarrollado por Año:")
for año, pct in resultado_bf['porcentaje_no_desarrollado'].items():
    print(f"  Año {año}: {pct*100:.2f}%")

print(f"\n\nReservas por Año de Origen:")
total_reserva_bf = 0
for año, reserva in resultado_bf['reservas'].items():
    print(f"  Año {año}: ${reserva:,.0f}")
    total_reserva_bf += reserva

print(f"\n{'='*70}")
print(f"RESERVA TOTAL (Bornhuetter-Ferguson): ${total_reserva_bf:,.0f}")
print(f"{'='*70}")

## 5. Método Bootstrap

El Bootstrap es un método estocástico que permite estimar la distribución de las reservas y calcular intervalos de confianza mediante simulación.

In [ ]:
# Aplicar Bootstrap (10,000 simulaciones)
print("⏳ Ejecutando Bootstrap con 10,000 simulaciones...")
print("   (Esto puede tomar unos segundos)\n")

bootstrap = Bootstrap(triangulo, n_simulaciones=10000, random_seed=42)
resultado_bootstrap = bootstrap.calcular_reservas()

print("📊 MÉTODO BOOTSTRAP")
print("="*70)

print(f"\nEstadísticas de Reserva Total:")
print(f"  Media:              ${resultado_bootstrap['reserva_media']:,.0f}")
print(f"  Mediana:            ${resultado_bootstrap['reserva_mediana']:,.0f}")
print(f"  Desviación Estándar: ${resultado_bootstrap['reserva_std']:,.0f}")
print(f"  Coef. Variación:    {resultado_bootstrap['coef_variacion']:.2%}")

print(f"\n\nIntervalos de Confianza:")
print(f"  75%: [${resultado_bootstrap['percentiles'][75]['lower']:,.0f}, "
      f"${resultado_bootstrap['percentiles'][75]['upper']:,.0f}]")
print(f"  90%: [${resultado_bootstrap['percentiles'][90]['lower']:,.0f}, "
      f"${resultado_bootstrap['percentiles'][90]['upper']:,.0f}]")
print(f"  95%: [${resultado_bootstrap['percentiles'][95]['lower']:,.0f}, "
      f"${resultado_bootstrap['percentiles'][95]['upper']:,.0f}]")
print(f"  99%: [${resultado_bootstrap['percentiles'][99]['lower']:,.0f}, "
      f"${resultado_bootstrap['percentiles'][99]['upper']:,.0f}]")

print(f"\n{'='*70}")
print(f"RESERVA TOTAL (Bootstrap - Media): ${resultado_bootstrap['reserva_media']:,.0f}")
print(f"{'='*70}")

In [ ]:
# Visualizar distribución del Bootstrap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Histograma de reservas simuladas
simulaciones = resultado_bootstrap['simulaciones']
ax1.hist(simulaciones, bins=50, color='#3498db', alpha=0.7, edgecolor='black')
ax1.axvline(resultado_bootstrap['reserva_media'], color='red', 
            linestyle='--', linewidth=2.5, label=f"Media: ${resultado_bootstrap['reserva_media']/1e6:.2f}M")
ax1.axvline(resultado_bootstrap['reserva_mediana'], color='orange', 
            linestyle='--', linewidth=2.5, label=f"Mediana: ${resultado_bootstrap['reserva_mediana']/1e6:.2f}M")

# Intervalos de confianza
p95_lower = resultado_bootstrap['percentiles'][95]['lower']
p95_upper = resultado_bootstrap['percentiles'][95]['upper']
ax1.axvline(p95_lower, color='green', linestyle=':', linewidth=2, alpha=0.7)
ax1.axvline(p95_upper, color='green', linestyle=':', linewidth=2, alpha=0.7, 
            label=f"IC 95%: [${p95_lower/1e6:.2f}M, ${p95_upper/1e6:.2f}M]")

ax1.set_xlabel('Reserva Total (MXN)', fontsize=12)
ax1.set_ylabel('Frecuencia', fontsize=12)
ax1.set_title('Distribución de Reservas - Bootstrap\n10,000 Simulaciones', 
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Gráfica 2: Q-Q Plot
from scipy import stats
stats.probplot(simulaciones, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot (Normal)\nVerificación de Normalidad', 
              fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Comparación de Métodos

Comparemos los resultados de los tres métodos para entender sus diferencias.

In [ ]:
# Tabla comparativa
comparacion_df = pd.DataFrame({
    'Método': ['Chain Ladder', 'Bornhuetter-Ferguson', 'Bootstrap (Media)', 'Bootstrap (P95 Upper)'],
    'Reserva Total': [
        total_reserva_cl,
        total_reserva_bf,
        resultado_bootstrap['reserva_media'],
        resultado_bootstrap['percentiles'][95]['upper']
    ],
    'Característica': [
        'Solo desarrollo histórico',
        'Desarrollo + prima esperada',
        'Estocástico - valor esperado',
        'Estocástico - conservador 95%'
    ]
})

comparacion_df['Diferencia vs CL'] = comparacion_df['Reserva Total'] - total_reserva_cl
comparacion_df['% Diferencia'] = (comparacion_df['Diferencia vs CL'] / total_reserva_cl * 100)

print("\n📊 COMPARACIÓN DE MÉTODOS DE RESERVAS")
print("="*100)
print(comparacion_df.to_string(index=False))

print(f"\n\n💡 ANÁLISIS:")
print(f"  - Chain Ladder tiende a ser más volátil para años recientes")
print(f"  - Bornhuetter-Ferguson es más estable al incorporar expectativas a priori")
print(f"  - Bootstrap proporciona distribución completa e intervalos de confianza")
print(f"  - Para solvencia se recomienda usar percentil 95% o superior")

In [ ]:
# Gráfica comparativa
fig, ax = plt.subplots(figsize=(14, 7))

metodos = ['Chain\nLadder', 'Bornhuetter-\nFerguson', 'Bootstrap\n(Media)', 'Bootstrap\n(P95)']
reservas_comp = [
    total_reserva_cl,
    total_reserva_bf,
    resultado_bootstrap['reserva_media'],
    resultado_bootstrap['percentiles'][95]['upper']
]
colores_comp = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

bars = ax.bar(metodos, reservas_comp, color=colores_comp, alpha=0.7, 
              edgecolor='black', linewidth=2)

# Agregar barras de error para Bootstrap
p75_lower = resultado_bootstrap['percentiles'][75]['lower']
p75_upper = resultado_bootstrap['percentiles'][75]['upper']
p95_lower = resultado_bootstrap['percentiles'][95]['lower']
p95_upper = resultado_bootstrap['percentiles'][95]['upper']

# Error bar para media
ax.errorbar(2, resultado_bootstrap['reserva_media'], 
            yerr=[[resultado_bootstrap['reserva_media'] - p75_lower], 
                  [p75_upper - resultado_bootstrap['reserva_media']]],
            fmt='none', color='black', linewidth=2, capsize=10, 
            label='IC 75% Bootstrap')

ax.set_ylabel('Reserva Total (MXN)', fontsize=13)
ax.set_title('Comparación de Métodos de Reservas Técnicas', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
ax.legend(fontsize=11)

# Agregar valores sobre las barras
for i, (bar, val) in enumerate(zip(bars, reservas_comp)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1e6:.2f}M',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Comparación completada")

## 7. Análisis de Sensibilidad

Analicemos cómo varían las reservas ante diferentes supuestos.

In [ ]:
# Sensibilidad del Loss Ratio en B-F
loss_ratios = [0.60, 0.65, 0.70, 0.75, 0.80]
reservas_bf_sens = []

print("📊 ANÁLISIS DE SENSIBILIDAD - Loss Ratio (B-F)")
print("="*70)

for lr in loss_ratios:
    bf_sens = BornhuetterFerguson(triangulo, primas_ganadas, lr)
    res_bf_sens = bf_sens.calcular_reservas()
    total_res = sum(res_bf_sens['reservas'].values())
    reservas_bf_sens.append(total_res)
    print(f"Loss Ratio {lr*100}%: ${total_res:,.0f}")

# Gráfica de sensibilidad
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot([lr*100 for lr in loss_ratios], reservas_bf_sens, 
        marker='o', linewidth=2.5, markersize=10, color='#e74c3c')
ax.axhline(total_reserva_cl, color='blue', linestyle='--', linewidth=2, 
           label=f'Chain Ladder: ${total_reserva_cl/1e6:.2f}M', alpha=0.7)

ax.set_xlabel('Loss Ratio Esperado (%)', fontsize=12)
ax.set_ylabel('Reserva Total (MXN)', fontsize=12)
ax.set_title('Sensibilidad de Reservas B-F al Loss Ratio Esperado', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Agregar valores
for lr, res in zip(loss_ratios, reservas_bf_sens):
    ax.text(lr*100, res, f'${res/1e6:.2f}M', ha='center', va='bottom', 
            fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## Conclusiones

En este notebook aprendimos:

1. **Chain Ladder**: Método determinístico basado en factores de desarrollo históricos
   - Ventajas: Simple, intuitivo, ampliamente usado
   - Desventajas: Volátil para años recientes, no considera información externa

2. **Bornhuetter-Ferguson**: Combina desarrollo observado con expectativas
   - Ventajas: Más estable, incorpora información de prima/siniestralidad esperada
   - Desventajas: Requiere estimar loss ratio a priori

3. **Bootstrap**: Método estocástico con distribución completa
   - Ventajas: Proporciona intervalos de confianza, mejor para análisis de riesgo
   - Desventajas: Computacionalmente intensivo, asume residuos independientes

### Recomendaciones:
- Usar múltiples métodos y comparar resultados
- Bootstrap para cuantificar incertidumbre y capital de solvencia
- B-F para años recientes con poco desarrollo
- Validar supuestos y realizar análisis de sensibilidad

### Próximos Pasos

Continúa con:
- **04_cumplimiento_cnsf.ipynb**: RCS, Reservas Técnicas regulatorias
- **05_validaciones_sat.ipynb**: Cumplimiento fiscal
- **06_reportes_cnsf.ipynb**: Reportes automatizados